# IMDb Sentiment Analysis (Bidirectional LSTM)

IMDb-এর movie review পড়ে model বলবে review-টা **positive** নাকি **negative**।

**এই version-এ তিনটা জিনিস ঠিক করা হয়েছে:**
1. **Validation আর test আলাদা।** আগে test set দিয়েই training-এর সময় validation হতো, তাই "accuracy" আসলে না-দেখা data-র ফল ছিল না। এখন training data-র 20% দিয়ে validation হয়, আর test set শুধু একদম শেষে একবার।
2. **Overfitting থামানো।** আগের run-এ epoch 2-এর পর validation loss বাড়ছিল (0.37 → 0.48)। এখন `EarlyStopping` সবচেয়ে ভালো epoch-এ থেমে সেই weight ফেরত নেয়।
3. **Padding-এর 0 উপেক্ষা করা।** `mask_zero=True` দেওয়ায় LSTM padding-এর 0-গুলো পড়ে না। আর বড় review-এর শুরুটা কাটা হয় (`truncating='pre'`), যাতে শেষের অংশটা থাকে, যেখানে সাধারণত রায় লেখা থাকে।

In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ১. ডেটা ডাউনলোড করা (শুধু সেরা ১০,০০০ শব্দ রাখব)
vocab_size = 10000
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=vocab_size)

print("Data Loaded Successfully! ✅")
print(f"Total Reviews for training: {len(X_train)}")
print(f"Total Reviews for testing:  {len(X_test)}")

In [ ]:
# একটা review আসলে কেমন দেখতে? সংখ্যাগুলোকে আবার শব্দে ফেরত নিই
word_index = imdb.get_word_index()
index_word = {i + 3: w for w, i in word_index.items()}   # ৩ ঘর সরানো (0, 1, 2 সংরক্ষিত)
index_word.update({0: "<PAD>", 1: "<START>", 2: "<UNK>"})

print("Numbers:", X_train[0][:12])
print("Words:  ", " ".join(index_word.get(i, "?") for i in X_train[0][:40]))
print("Label:  ", "Positive 😊" if y_train[0] == 1 else "Negative 😡")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Review কত লম্বা? এটা দেখেই ঠিক করবো সব review কত শব্দে কাটবো
lengths = np.array([len(r) for r in X_train])
print(f"গড় দৈর্ঘ্য: {lengths.mean():.0f} শব্দ | Median: {np.median(lengths):.0f} | সবচেয়ে বড়: {lengths.max()}")
print(f"২৫০ শব্দ বা তার কম: {(lengths <= 250).mean() * 100:.1f}% review")
print(f"Positive: {int(y_train.sum())} | Negative: {int(len(y_train) - y_train.sum())}")

plt.figure(figsize=(10, 4))
plt.hist(lengths, bins=60, range=(0, 1200), color='steelblue')
plt.axvline(250, color='red', linestyle='--', label='max_length = 250')
plt.title('Review Length Distribution')
plt.xlabel('Words per review')
plt.ylabel('Number of reviews')
plt.legend()
plt.show()

In [ ]:
# আমরা সব রিভিউ ২৫০ শব্দের বানাব
max_length = 250

# ছোট রিভিউর শেষে '0' বসবে (padding='post')। Model-এ mask_zero=True থাকায় এই 0-গুলো পড়া হবে না।
# বড় রিভিউর শুরুটা কাটা হবে (truncating='pre'), যাতে শেষের অংশ, যেখানে সাধারণত রায় থাকে, থেকে যায়।
X_train = pad_sequences(X_train, maxlen=max_length, padding='post', truncating='pre')
X_test = pad_sequences(X_test, maxlen=max_length, padding='post', truncating='pre')

print(f"New Shape of Data: {X_train.shape}")
print("All reviews are now equal size! 📏")

In [ ]:
# আপগ্রেডেড আর্কিটেকচার (Bidirectional LSTM)
model = tf.keras.models.Sequential([
    # ১. Embedding: প্রতিটা শব্দকে ৩২টা সংখ্যার vector বানায়।
    #    mask_zero=True: '0' মানে padding, পরের layer-গুলো এটা উপেক্ষা করবে।
    tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=32, mask_zero=True),

    # ২. Bidirectional LSTM (The Upgrade)
    # এটা পড়ার সময় দুই দিক থেকেই কনটেক্সট নেবে (বাম-ডান এবং ডান-বাম)
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64)),

    # ৩. একটু এক্সট্রা ব্রেইন পাওয়ার (Dense Layer)
    tf.keras.layers.Dense(64, activation='relu'),

    # ৪. ড্রপআউট (যাতে মুখস্থ না করে - রিসার্চারদের ট্রিক)
    tf.keras.layers.Dropout(0.5),

    # ৫. আউটপুট
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# কম্পাইল করা
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.build(input_shape=(None, max_length))  # summary-তে parameter সংখ্যা দেখানোর জন্য
print("Upgraded Bidirectional Model Built! 🚀")
model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

print("Training Started... Please wait ⏳")

# Validation: training data-র শেষ 20% (৫,০০০ review)। Test set এখানে একবারও ব্যবহার হচ্ছে না।
# EarlyStopping: validation loss পরপর ২ epoch না কমলে থেমে যাবে, আর সবচেয়ে ভালো epoch-এর weight ফেরত নেবে।
early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

history = model.fit(X_train, y_train,
                    epochs=10,
                    validation_split=0.2,
                    callbacks=[early_stop])

best_epoch = min(range(len(history.history['val_loss'])), key=history.history['val_loss'].__getitem__) + 1
print(f"Training Finished! 🎉 Best epoch: {best_epoch}")

In [ ]:
# চূড়ান্ত পরীক্ষা: ২৫,০০০ না-দেখা review, পুরো notebook-এ শুধু এই একবার
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Loss:     {test_loss:.4f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

# ১. একুরেন্সি গ্রাফ (Accuracy)
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy', marker='o')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', marker='o')
plt.title('How Smart is the Model?')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# ২. লস গ্রাফ (Loss/Mistakes): validation loss বাড়তে শুরু করলে বুঝবে overfitting শুরু হয়েছে
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss', marker='o')
plt.plot(history.history['val_loss'], label='Validation Loss', marker='o')
plt.title('Mistake Rate')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import numpy as np

# ১. টেস্ট ডেটার ওপর প্রেডিকশন নেওয়া
y_pred = model.predict(X_test)
y_pred_classes = (y_pred > 0.5).astype("int32").ravel()  # ০.৫ এর বেশি হলে ১, কম হলে ০

# ২. কনফিউশন ম্যাট্রিক্স তৈরি
cm = confusion_matrix(y_test, y_pred_classes)

# ৩. সুন্দর করে প্লট করা
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix: True vs Predicted')
plt.show()

# ৪. প্রতিটা class-এর precision, recall আর F1
print(classification_report(y_test, y_pred_classes, target_names=['Negative', 'Positive'], digits=4))

In [ ]:
import re

# ১. ডিকশনারি (ইংরেজি শব্দ -> সংখ্যা) উপরের cell-এই নামানো হয়েছে: word_index

def encode_review(text):
    # ছোট হাতের অক্ষর করে শব্দ আলাদা করা; দাঁড়ি-কমা বাদ ("time," -> "time")
    words = re.findall(r"[a-z0-9']+", text.lower())

    # Training data-র প্রতিটা review '1' (start token) দিয়ে শুরু হয়, তাই এখানেও তাই
    encoded = [1]
    for word in words:
        idx = word_index.get(word, -1) + 3   # ৩ ঘর পিছিয়ে সেট করা (0, 1, 2 আগে থেকেই সংরক্ষিত)
        if idx < 3 or idx >= vocab_size:     # ডিকশনারিতে নেই, বা ১০,০০০ লিমিটের বাইরে
            idx = 2                          # তাহলে 'Unknown' (2)
        encoded.append(idx)

    # Training-এর মতো একই নিয়মে padding
    return pad_sequences([encoded], maxlen=max_length, padding='post', truncating='pre')

def predict_sentiment(text):
    prediction = model.predict(encode_review(text), verbose=0)[0][0]

    # রেজাল্ট দেখানো
    print(f"\nReview: '{text}'")
    print(f"Confidence Score: {prediction:.4f}")  # (০ থেকে ১ এর মধ্যে)

    if prediction > 0.5:
        print("Result: POSITIVE Review! (ভালো) 😊")
    else:
        print("Result: NEGATIVE Review! (বাজে) 😡")

# ২. চলো টেস্ট করি!
predict_sentiment("This movie is amazing and I loved it")
predict_sentiment("Worst movie ever, total waste of time")
predict_sentiment("The visual effects were stunning, but the plot was nonexistent.")
predict_sentiment("The acting was good but the story was total garbage")
predict_sentiment("This movie was not bad at all")  # negation: খারাপ শব্দ, কিন্তু অর্থ ভালো
predict_sentiment("The movie was so long that I finished my graduation while watching it")  # sarcasm

In [ ]:
# মডেল সেভ করা (নতুন .keras ফরম্যাটে; পুরনো .h5 এখন legacy)
model.save('sentiment_model_bilstm.keras')
print("Model Saved Successfully! ✅")